In [4]:
# =============================================================================
# SCRIPT DE DOWNLOAD AUTOMÁTICO (Refatorado - SOLID)
# Foco: PAM, PPM (bovinos), PIB Municipal
# =============================================================================

import os
import pandas as pd
import sidrapy
from pathlib import Path
from tqdm import tqdm
import warnings
from abc import ABC, abstractmethod
from typing import List
import time

warnings.filterwarnings("ignore", category=UserWarning)


# ==================== CONFIGURAÇÃO DE PASTAS ====================
class CaminhosDados:
    DATA_DIR = Path("../../data")
    LANDING_DIR = DATA_DIR / "landing"
    BRONZE_DIR = DATA_DIR / "01_bronze"

    @classmethod
    def inicializar_pastas(cls):
        for p in [cls.LANDING_DIR, cls.BRONZE_DIR]:
            p.mkdir(parents=True, exist_ok=True)


# ==================== ARMAZENAMENTO ====================
class EscritorDeDados(ABC):
    @abstractmethod
    def escrever(
        self, df: pd.DataFrame, nome_dataset: str, particionar_por: List[str] = None
    ):
        pass


class EscritorParquet(EscritorDeDados):
    def __init__(self, diretorio_base: Path):
        self.diretorio_base = diretorio_base

    def escrever(
        self, df: pd.DataFrame, nome_dataset: str, particionar_por: List[str] = None
    ):
        caminho_saida = self.diretorio_base / nome_dataset
        caminho_saida.mkdir(parents=True, exist_ok=True)

        opcoes_escrita = {"engine": "pyarrow", "compression": "snappy", "index": False}

        particionar_por_real = [c for c in (particionar_por or []) if c in df.columns]

        if particionar_por_real:
            df.to_parquet(
                caminho_saida, partition_cols=particionar_por_real, **opcoes_escrita
            )
        else:
            df.to_parquet(caminho_saida / f"{nome_dataset}.parquet", **opcoes_escrita)

        print(
            f"✔ {nome_dataset} → {len(df):,} linhas | Particionado por: {particionar_por_real or 'nenhum'}"
        )


# ==================== EXTRAÇÃO ====================
class ExtratorDeDados(ABC):
    @abstractmethod
    def extrair(self) -> pd.DataFrame:
        pass


class ExtratorSidra(ExtratorDeDados):
    def __init__(self, escritor: EscritorDeDados):
        self.escritor = escritor

    def _padronizar_municipio(
        self, df: pd.DataFrame, coluna_id: str = "D2C"
    ) -> pd.DataFrame:
        if coluna_id in df.columns:
            df["codigo_municipio"] = (
                df[coluna_id].astype(str).str.strip().str.zfill(7).str[-7:]
            )
        return df

    def _garantir_coluna_valor(self, df: pd.DataFrame) -> pd.DataFrame:
        if "V" in df.columns:
            df = df.rename(columns={"V": "valor"})
        elif any(c.startswith("V") for c in df.columns if c != "V"):
            col_val = next((c for c in df.columns if c.startswith("V")), None)
            if col_val:
                df = df.rename(columns={col_val: "valor"})
        return df


class ExtratorPAM(ExtratorSidra):
    def __init__(self, anos: List[int], escritor: EscritorDeDados):
        super().__init__(escritor)
        self.anos = [a for a in anos if a <= 2024]  # 2025 provavelmente ainda não tem

    def extrair(self) -> pd.DataFrame:
        def buscar_tabela(codigo_tabela: str, ano: int, tipo: str) -> pd.DataFrame:
            try:
                time.sleep(0.6)  # evita bloqueio do servidor SIDRA
                dados = sidrapy.get_table(
                    table_code=codigo_tabela,
                    territorial_level="6",
                    ibge_territorial_code="all",
                    period=str(ano),
                )
                if dados is None or dados.empty:
                    return pd.DataFrame()

                dados["ano"] = ano
                dados["tipo_lavoura"] = tipo
                return dados
            except Exception as e:
                print(f"Erro PAM {tipo} {ano}: {str(e)[:80]}...")
                return pd.DataFrame()

        lista_dfs = []
        for ano in tqdm(self.anos, desc="Extraindo PAM"):
            lista_dfs.append(buscar_tabela("1612", ano, "temporaria"))
            lista_dfs.append(buscar_tabela("1613", ano, "permanente"))

        if not lista_dfs:
            return pd.DataFrame()

        df_completo = pd.concat(
            [df for df in lista_dfs if not df.empty], ignore_index=True
        )
        return self._preprocessar(df_completo)

    def _preprocessar(self, df: pd.DataFrame) -> pd.DataFrame:
        mapa_renomear = {
            "D1C": "ano",
            "D2C": "cd_mun",
            "D3N": "produto",
            "D4N": "medida",
            "V": "valor",
        }
        df = df.rename(columns=mapa_renomear)
        df = self._padronizar_municipio(df, "cd_mun")
        df = self._garantir_coluna_valor(df)

        colunas_desejadas = [
            "ano",
            "codigo_municipio",
            "produto",
            "medida",
            "valor",
            "tipo_lavoura",
        ]
        cols_ok = [c for c in colunas_desejadas if c in df.columns]
        return df[cols_ok].drop_duplicates().copy()


class ExtratorPPM(ExtratorSidra):
    def __init__(self, anos: List[int], escritor: EscritorDeDados):
        super().__init__(escritor)
        self.anos = [a for a in anos if a <= 2024]

    def extrair(self) -> pd.DataFrame:
        lista_dfs = []
        for ano in tqdm(self.anos, desc="Extraindo PPM Bovinos"):
            try:
                time.sleep(0.6)
                df = sidrapy.get_table(
                    table_code="3939",
                    territorial_level="6",
                    ibge_territorial_code="all",
                    period=str(ano),
                    variable="3135",  # Efetivo do rebanho (cabeças) - mais usado para bovinos
                    classification="79/100",  # 79 = Grande grupo / 100 = Bovinos
                )
                if df is not None and not df.empty:
                    df["ano"] = ano
                    lista_dfs.append(df)
            except Exception as e:
                print(f"Erro PPM {ano}: {str(e)[:80]}...")

        if not lista_dfs:
            return pd.DataFrame()

        df_completo = pd.concat(lista_dfs, ignore_index=True)
        df_completo = self._padronizar_municipio(df_completo)
        df_completo = self._garantir_coluna_valor(df_completo)
        return df_completo


class ExtratorPIB(ExtratorSidra):
    def extrair(self) -> pd.DataFrame:
        try:
            time.sleep(0.8)
            df = sidrapy.get_table(
                table_code="5938",
                territorial_level="6",
                ibge_territorial_code="all",
                period="all",
            )
            if df is None or df.empty:
                print("PIB: tabela vazia ou falha na requisição")
                return pd.DataFrame()

            df = df.rename(
                columns={
                    "D1C": "ano",
                    "V": "valor",
                    "D3N": "indicador",
                    "D4N": "medida",
                }
            )
            df = self._padronizar_municipio(df)
            df = self._garantir_coluna_valor(df)

            # Filtro mais amplo para atividades agropecuárias
            mask = df["indicador"].str.contains(
                "agro|agricultura|pecuária|silvicultura|pesca|florestal",
                case=False,
                na=False,
                regex=True,
            )
            df_filtrado = df[mask].copy()
            if df_filtrado.empty:
                print("PIB: nenhum registro agro encontrado após filtro")
            return df_filtrado
        except Exception as e:
            print(f"Erro PIB: {str(e)[:120]}")
            return pd.DataFrame()


# ==================== MOTOR DE ORQUESTRAÇÃO ====================
class OrquestradorDownload:
    def __init__(self, escritor: EscritorDeDados):
        self.escritor = escritor

    def executar(self, extratores: dict):
        for nome_dataset, extrator in extratores.items():
            print(f"\n📥 Iniciando: {nome_dataset}")
            df = extrator.extrair()
            if not df.empty:
                particoes = ["ano"] if "ano" in df.columns else None
                self.escritor.escrever(df, nome_dataset, particoes)
            else:
                print(f"⚠️ Nenhum dado para {nome_dataset}")


# ==================== EXECUÇÃO PRINCIPAL ====================
if __name__ == "__main__":
    CaminhosDados.inicializar_pastas()

    escritor = EscritorParquet(CaminhosDados.BRONZE_DIR)
    orquestrador = OrquestradorDownload(escritor)

    # Sugestão: comece com poucos anos para testar
    anos_alvo = list(range(2010, 2025))  # até 2024 é mais seguro

    servicos_extracao = {
        "pam": ExtratorPAM(anos_alvo, escritor),
        "ppm_bovinos": ExtratorPPM(anos_alvo, escritor),
        "pib_vab_agro": ExtratorPIB(escritor),
    }

    orquestrador.executar(servicos_extracao)

    print("\n✅ Processamento concluído!")

    # ==================== BASES AMBIENTAIS - DOWNLOAD MANUAL ====================
    print("\n" + "=" * 70)
    print(" BASES AMBIENTAIS → DOWNLOAD MANUAL (sem API estável)")
    print("=" * 70)
    print(
        """
1. PRODES (desmatamento Amazônia Legal)
   → https://terrabrasilis.dpi.inpe.br/downloads/
   → "PRODES por município" → data/landing/prodes_municipios.csv

2. DETER (alertas)
   → Mesmo site → pasta DETER

3. MapBiomas (coleção mais recente)
   → https://brasil.mapbiomas.org/estatisticas
   → Estatísticas por Município → data/landing/mapbiomas_municipios_cX.csv

4. Embargos IBAMA
   → https://dadosabertos.ibama.gov.br/dataset/termos-de-embargo

5. Comex Stat (soja/carne)
   → https://comexstat.mdic.gov.br/pt/geral
   → Filtro NCM soja/carne → exportar CSV
"""
    )
    print("\nApós baixar → converter manualmente para Parquet:")
    print(
        "df = pd.read_csv('arquivo.csv', encoding='latin1', sep=';', low_memory=False)"
    )
    print("df.to_parquet('data/bronze/nome.parquet', index=False)\n")


📥 Iniciando: pam


Extraindo PAM:   7%|▋         | 1/15 [01:23<19:29, 83.57s/it]


KeyboardInterrupt: 